# Behaviour-Based Detection of Living-off-the-Land Attacks via Process-Provenance Telemetry
**IT 7178 — Research Progress: Implementation & Results (executed notebook)**
Olaoluwa E. Sipeolu · Georgia Southern University

Two experiments:
1. **Linux lab (auditd).** Routine administrative tasks versus emulated LotL techniques mapped to
   MITRE ATT&CK, captured on an Ubuntu Server VM.
2. **Windows pilot (Sysmon, OTRF Security Datasets).** Real ATT&CK technique executions recorded
   on a lab domain, together with the normal background activity of the same hosts.

In both, a provenance-aware behaviour score is compared with a signature-style
**name-based baseline** that flags any well-known LotL binary.

## 0. Setup

In [ ]:
import os, sys, glob, zipfile
if not os.path.isdir("src"):                       # Colab: fetch the project repository
    os.system("git clone -q https://github.com/os01752/lotl-provenance-detection.git repo")
    os.chdir("repo")
sys.path.insert(0, "src")
import provenance as P, sysmon as W
import pandas as pd, numpy as np, networkx as nx, matplotlib.pyplot as plt
from sklearn.metrics import (precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, roc_curve)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 70)

# unzip the OTRF pilot datasets (stored in data/otrf, see data/otrf/README.md)
for z in glob.glob("data/otrf/*.zip"):
    d = os.path.join("data/otrf/json", os.path.basename(z)[:-4])
    if not os.path.isdir(d):
        os.makedirs(d); zipfile.ZipFile(z).extractall(d)
print("Linux captures:", len(glob.glob("captures/*.log")),
      "| OTRF datasets:", len(glob.glob("data/otrf/json/*/*.json")))

## 1. Experiment 1 — Linux lab (auditd)
Each `execve` is a node. A process that replaces its image (bash exec'ing its last command) is an
*exec* edge; a child links to its parent's latest execution (*fork* edge). Failed `execve` calls
(PATH lookups) are dropped and only the task's own process tree is kept.

In [ ]:
LF, LG, LD = P.load_all("captures")
LF["baseline"] = [P.baseline_name_detector(LG[n]) for n in LF.capture]
print(LF.label.value_counts().to_string(), "\nhard negatives:", int(LF.hard_negative.sum()))
LF[["capture","label","technique","n_processes","chain_depth","lotl_binaries","behaviour_score","baseline"]]

### 1.1 Behavioural indicators per capture

In [ ]:
ind = LF.set_index("capture")[P.INDICATORS].rename(columns=lambda c: c[2:])
ind = ind.loc[LF.sort_values(["label","capture"]).capture]
fig, ax = plt.subplots(figsize=(9, max(4, 0.24*len(ind))))
ax.imshow(ind.values, aspect="auto", cmap="Reds", vmin=0, vmax=1.4)
ax.set_xticks(range(ind.shape[1])); ax.set_xticklabels(ind.columns, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(ind))); ax.set_yticklabels(ind.index, fontsize=7)
nb = int((LF.label=="benign").sum()); ax.axhline(nb-0.5, color="black", lw=1.5)
ax.set_title("Behavioural indicators (rows above line: benign; below: emulated LotL)", fontsize=10)
plt.tight_layout(); plt.savefig("fig_indicators.png", dpi=160, bbox_inches="tight"); plt.show()

### 1.2 Provenance graphs: hard negatives vs. attacks using the same tools

In [ ]:
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 7})
GREEN, RED, GREY = "#2e8b57", "#b03a2e", "#d9d9d9"

def draw(ax, g, title, mal, ctx_key=None, fs=6.3):
    """Print-size provenance tree: each process is a labelled box; arrows point parent -> child."""
    pos = P.tree_layout(g, x_gap=1.0, y_gap=0.9)
    nx.draw_networkx_edges(g, pos, ax=ax, arrows=True, arrowsize=6, width=0.7, edge_color="#444",
                           arrowstyle="-|>", node_size=420, min_target_margin=7, min_source_margin=7)
    for n, (x, y) in pos.items():
        ctx = bool(ctx_key and g.nodes[n].get(ctx_key))
        c = g.nodes[n]["comm"].removesuffix(".exe")
        c = c if len(c) <= 11 else c[:10] + "…"
        ax.text(x, y, c, ha="center", va="center", fontsize=fs, color="black" if ctx else "white",
                fontweight="bold", zorder=3,
                bbox=dict(boxstyle="round,pad=0.28", fc=GREY if ctx else (RED if mal else GREEN), ec="black", lw=0.5))
    xs = [x for x, _ in pos.values()] or [0]; ys = [y for _, y in pos.values()] or [0]
    ax.set_xlim(min(xs) - 0.7, max(xs) + 0.7); ax.set_ylim(min(ys) - 0.5, max(ys) + 0.5)
    ax.set_title(title, fontsize=7); ax.axis("off")

pick = lambda c: [x for x in c if x in LG][:3]
top = pick(["benign_hard_curl_download","benign_hard_chmod_script","benign_hard_nested_shell","benign_sysinfo","benign_netcheck","benign_useradmin"])
bot = pick(["lotl_T1105_curl_hidden_exec","lotl_T1222.002_chmod_hidden_exec","lotl_T1027_hex_xxd_bash","lotl_download_exec","lotl_obfusc_shell","lotl_cron_persist"])
nice = lambda n: n.replace("benign_hard_","").replace("benign_","").replace("lotl_","").replace("_"," ")
fig, axes = plt.subplots(2, 3, figsize=(7.16, 3.1))
for i, n in enumerate(top): draw(axes[0][i], LG[n], f"({'abc'[i]}) benign: {nice(n)}", False)
for i, n in enumerate(bot): draw(axes[1][i], LG[n], f"({'def'[i]}) LotL: {nice(n)}", True)
for ax in list(axes[0][len(top):]) + list(axes[1][len(bot):]): ax.axis("off")
plt.tight_layout(pad=0.3, h_pad=0.6); plt.savefig("fig_linux_graphs.png", dpi=300, bbox_inches="tight"); plt.show()

# caption text that matches the panels actually shown (written to the report)
hard = all(n.startswith("benign_hard_") for n in top)
cap = ("Linux process-provenance graphs. (a)--(c) benign administration"
       + (" hard negatives that use the same binaries as the attacks (\\texttt{curl}, \\texttt{chmod}, nested shells)" if hard else "")
       + "; (d)--(f) emulated LotL techniques. Boxes are process executions; arrows point from parent to child."
       + (" The attacks differ by executing a hidden payload in \\texttt{/tmp} or piping decoded data into a shell." if hard else ""))
os.makedirs("report", exist_ok=True)
open("report/fig_captions.tex", "w").write("\\newcommand{\\LinuxFigCaption}{" + cap + "}\n")
print(cap)

## 2. Experiment 2 — Windows pilot (Sysmon EID 1, OTRF Security Datasets)
Unit of analysis: an *execution unit* — a process whose parent started before the capture window,
plus its descendants, with that parent attached as a context node (grey). Ground truth: a unit is
malicious if it contains the technique documented for that OTRF dataset; side-effect units of the
payload (UWP Calculator activation) are excluded.

**Caveat:** the Windows indicator vocabulary was written after inspecting these 13 files, so this is
a feasibility pilot. The final evaluation freezes the indicators and uses unseen OTRF datasets.

In [ ]:
WF, WG = W.load_otrf("data/otrf/json")
print(WF.label.value_counts().to_string())
WF[["unit","label","technique","chain","behaviour_score","baseline"]].sort_values(["label","unit"])

In [ ]:
wshow = [("empire_launcher_sct_regsvr32", "regsvr32"), ("empire_wmic_add_user_backdoor", "net.exe"),
         ("psh_cmstp_execution_bypassuac", "dllhost"), ("schtask_create", "cscript")]
fig, axes = plt.subplots(2, 2, figsize=(3.5, 3.9))
for i, (ax, (ds, key)) in enumerate(zip(axes.flat, wshow)):
    cand = [u for u in WG if u.startswith(ds) and any(key in WG[u].nodes[n]["comm"] for n in WG[u].nodes)]
    if not cand: ax.axis("off"); continue
    u = cand[0]; row = WF.set_index("unit").loc[u]; mal = row["label"] == "malicious"
    draw(ax, WG[u], f"({'abcd'[i]}) " + (row["technique"] if mal else "benign"), mal, "context", fs=5.6)
plt.tight_layout(pad=0.3, h_pad=0.8); plt.savefig("fig_windows_graphs.png", dpi=300, bbox_inches="tight"); plt.show()

## 3. Results: baseline vs. provenance score

In [ ]:
def metrics(y, p, s=None):
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    return dict(TP=tp, FP=fp, FN=fn, TN=tn,
                precision=round(precision_score(y, p, zero_division=0), 3),
                recall=round(recall_score(y, p, zero_division=0), 3),
                F1=round(f1_score(y, p, zero_division=0), 3),
                FPR=round(fp / max(fp + tn, 1), 3))

rows = []
for name, F in [("Linux lab", LF), ("Windows pilot", WF)]:
    y = (F.label == "malicious").astype(int).values
    rows.append(dict(dataset=name, detector="Name-based baseline", **metrics(y, F.baseline.values)))
    for t in (1, 2):
        rows.append(dict(dataset=name, detector=f"Provenance score >= {t}",
                         **metrics(y, (F.behaviour_score >= t).astype(int).values)))
    rows.append(dict(dataset=name, detector="Provenance score ROC-AUC",
                     TP="", FP="", FN="", TN="", precision="", recall="",
                     F1=round(roc_auc_score(y, F.behaviour_score), 3), FPR=""))
results = pd.DataFrame(rows); results.to_csv("results_table.csv", index=False); results

### 3.0 LaTeX tables for the report (written to `report/`)

In [ ]:
os.makedirs("report", exist_ok=True)
nl = (LF.label=="malicious").sum(); bl = (LF.label=="benign").sum(); hn = int(LF.hard_negative.sum())
nw = (WF.label=="malicious").sum(); bw = (WF.label=="benign").sum()
tl = LF[LF.label=="malicious"].technique.nunique(); tw = WF[WF.label=="malicious"].technique.nunique()
ds = rf"""\begin{{table}}[t]
\caption{{Datasets}}\label{{tab:datasets}}\centering\footnotesize
\resizebox{{\columnwidth}}{{!}}{{\begin{{tabular}}{{llrrr}}\toprule
\textbf{{Dataset}} & \textbf{{Unit}} & \textbf{{Benign}} & \textbf{{Malicious}} & \textbf{{ATT\&CK IDs}}\\\midrule
Linux lab (auditd) & activity & {bl} & {nl} & {tl}\\
Windows pilot (OTRF Sysmon) & execution unit & {bw} & {nw} & {tw}\\
\bottomrule\end{{tabular}}}}\end{{table}}
"""
open("report/dataset_table.tex","w").write(ds)
lines = []
for dsn in ["Linux lab", "Windows pilot"]:
    sub = results[results.dataset==dsn]
    auc = sub[sub.detector.str.contains("AUC")].F1.iloc[0]
    body = sub[~sub.detector.str.contains("AUC")]
    lines.append(rf"\multicolumn{{6}}{{l}}{{\textit{{{dsn}}} (ROC-AUC of score = {auc:.3f})}}\\")
    for r in body.itertuples():
        lines.append(rf"\quad {r.detector.replace('>=', '$\\ge$')} & {r.precision:.3f} & {r.recall:.3f} & {r.F1:.3f} & {r.FPR:.3f} & {r.TP}/{r.FP}/{r.FN}/{r.TN}\\")
rt = (r"\begin{table}[t]" "\n" r"\caption{Detection performance: name-based baseline vs. provenance behaviour score}\label{tab:results}" "\n"
      r"\centering\footnotesize\setlength{\tabcolsep}{3.5pt}" "\n" r"\resizebox{\columnwidth}{!}{\begin{tabular}{lccccc}\toprule" "\n"
      r"\textbf{Detector} & \textbf{P} & \textbf{R} & \textbf{F$_1$} & \textbf{FPR} & \textbf{TP/FP/FN/TN}\\\midrule" "\n"
      + "\n".join(lines) + "\n" r"\bottomrule\end{tabular}}\end{table}" "\n")
open("report/results_table.tex","w").write(rt)
print(ds); print(rt)

### 3.0b Numbers quoted in the report text (written to `report/numbers.tex`)

In [ ]:
def m(F, pred):
    y = (F.label == "malicious").astype(int); p = pred.astype(int)
    fp = int(((p == 1) & (y == 0)).sum()); tp = int(((p == 1) & (y == 1)).sum())
    return dict(fp=fp, tp=tp, fpr=fp / max(int((y == 0).sum()), 1), rec=tp / max(int(y.sum()), 1),
                prec=tp / max(tp + fp, 1))
def nice(names):
    names = [n.replace("benign_hard_", "").replace("benign_", "").replace("lotl_", "").replace("_", r"\_") for n in names]
    return ", ".join(r"\texttt{" + n + "}" for n in names) if names else "none"
hnF = LF[LF.hard_negative == 1]
lb, l1, l2 = m(LF, LF.baseline), m(LF, LF.behaviour_score >= 1), m(LF, LF.behaviour_score >= 2)
wb, w1, w2 = m(WF, WF.baseline), m(WF, WF.behaviour_score >= 1), m(WF, WF.behaviour_score >= 2)
yL = (LF.label == "malicious").astype(int); yW = (WF.label == "malicious").astype(int)
vals = dict(
    LinTotal=len(LF), LinBenign=int((LF.label == "benign").sum()), LinMal=int(yL.sum()),
    LinHard=len(hnF), LinTech=LF[LF.label == "malicious"].technique.nunique(),
    LinBaseFP=lb["fp"], LinBaseFPR=f"{lb['fpr']:.2f}", LinBasePrec=f"{lb['prec']:.2f}", LinBaseRec=f"{lb['rec']:.2f}",
    LinScoreFP=l1["fp"], LinScoreFPR=f"{l1['fpr']:.2f}", LinScorePrec=f"{l1['prec']:.2f}", LinScoreRec=f"{l1['rec']:.2f}",
    LinScoreTwoRec=f"{l2['rec']:.2f}", LinAUC=f"{roc_auc_score(yL, LF.behaviour_score):.2f}",
    LinHardBase=int(hnF.baseline.sum()), LinHardScore=int((hnF.behaviour_score >= 1).sum()),
    LinMissed=nice(list(LF[(yL == 1) & (LF.behaviour_score < 1)].capture)),
    LinMissedTwo=nice(list(LF[(yL == 1) & (LF.behaviour_score < 2)].capture)),
    LinBaseMissed=nice(list(LF[(yL == 1) & (LF.baseline == 0)].capture)),
    WinTotal=len(WF), WinBenign=int((WF.label == "benign").sum()), WinMal=int(yW.sum()),
    WinTech=WF[WF.label == "malicious"].technique.nunique(),
    WinBaseFP=wb["fp"], WinBaseRec=f"{wb['rec']:.2f}", WinScoreFP=w1["fp"], WinScoreRec=f"{w1['rec']:.2f}",
    WinScoreTwoRec=f"{w2['rec']:.2f}", WinAUC=f"{roc_auc_score(yW, WF.behaviour_score):.2f}",
)
open("report/numbers.tex", "w").write("".join(f"\\newcommand{{\\{k}}}{{{v}}}\n" for k, v in vals.items()))
pd.Series(vals)

### 3.1 Per-activity detail (Linux)

In [ ]:
LF[["capture", "label", "lotl_binaries", "baseline", "behaviour_score"] + P.INDICATORS]

### 3.2 Score distributions and ROC

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(7.16, 2.3))
for i, (ax, (name, F)) in enumerate(zip(axes[:2], [("Linux lab", LF), ("Windows pilot", WF)])):
    mx = int(max(F.behaviour_score.max(), 1))
    bins = np.arange(-0.5, mx + 1.5)
    ax.hist(F[F.label=="benign"].behaviour_score, bins=bins, alpha=.8, color=GREEN, label="benign")
    ax.hist(F[F.label=="malicious"].behaviour_score, bins=bins, alpha=.8, color=RED, label="malicious")
    ax.set_title(f"({'ab'[i]}) {name}", fontsize=7); ax.set_xlabel("indicators fired, $S(G)$"); ax.set_ylabel("count")
    ax.legend(fontsize=6, frameon=False)
ax = axes[2]
for name, F, c in [("Linux lab", LF, "#2c3e50"), ("Windows pilot", WF, "#8e44ad")]:
    y = (F.label=="malicious").astype(int)
    fpr, tpr, _ = roc_curve(y, F.behaviour_score)
    ax.plot(fpr, tpr, marker="o", ms=3, lw=1, color=c, label=f"{name} score (AUC={roc_auc_score(y, F.behaviour_score):.2f})")
    yb = F.baseline; ax.scatter([((yb==1)&(y==0)).sum()/max((y==0).sum(),1)], [((yb==1)&(y==1)).sum()/max(y.sum(),1)],
                                marker="X", s=40, color=c, label=f"{name} baseline")
ax.plot([0,1],[0,1],"--",color="grey",lw=.6); ax.set_xlabel("false-positive rate"); ax.set_ylabel("true-positive rate")
ax.set_title("(c) ROC", fontsize=7); ax.legend(fontsize=5.5, frameon=False, loc="lower right")
plt.tight_layout(pad=0.3); plt.savefig("fig_results.png", dpi=300, bbox_inches="tight"); plt.show()

## 4. Error analysis

In [ ]:
for name, F, key in [("Linux", LF, "capture"), ("Windows", WF, "unit")]:
    for t in (1,):
        fp = F[(F.label=="benign") & (F.behaviour_score>=t)]
        fn = F[(F.label=="malicious") & (F.behaviour_score<t)]
        print(f"--- {name}: false positives at score>={t}:", list(fp[key]))
        print(f"--- {name}: missed attacks at score>={t}:", list(fn[key]))
    fpb = F[(F.label=="benign") & (F.baseline==1)]
    print(f"--- {name}: baseline false positives:", list(fpb[key]))

## 5. Summary and limitations
- The provenance pipeline works end-to-end on both Linux auditd and Windows Sysmon telemetry using one
  graph representation.
- On the initial Linux lab set, both detectors separate the classes: the benign tasks do not use
  attack-associated binaries, so this set cannot yet distinguish the two approaches. Adding benign
  administration that uses the same binaries as the attacks is the next data-collection step.
- On the Windows pilot the baseline's false positives come from legitimate script-host use
  (`MonitoringHost -> cscript`); the behaviour score's errors are an encoded-looking Windows Search
  argument (FP) and an injection target that looks benign in isolation (FN).
- **Limitations:** small, controlled datasets; indicators designed by the author (Linux) or after
  inspecting the pilot files (Windows); rule-based score without learned weights.
- **Next (final paper):** freeze indicators and evaluate on unseen OTRF datasets; add a learned
  model with cross-validation; obfuscation-robustness experiment; more varied benign activity.